# 05 • ค้นหา SOP ด้วย Semantic Search

เป้าหมาย: สร้าง Embedding ของข้อความงานรับซื้อ QC คลังสินค้า และขนส่ง แล้วจัดอันดับด้วย cosine similarity ยังไม่ให้ LLM สร้างคำตอบ

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ SOP ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

บริบทงานอ้างอิงจาก [เว็บไซต์กลุ่มศรีตรัง](https://www.sritranggroup.com/th/home) ส่วนรูปแบบ notebook อ้างอิงจาก [ตัวอย่าง GitHub](https://github.com/biodatlab/cpf-genai-workshop/tree/main/tutorial_notebooks) และใช้ [OpenAI API documentation](https://developers.openai.com/api/docs) สำหรับ syntax

## เตรียมพร้อมก่อนเริ่ม
เพิ่ม Secret ชื่อ `OPENAI_API_KEY` ใน Colab ด้วยตนเอง และเปิด **Notebook access** สำหรับ notebook นี้ โค้ดจะอ่านผ่าน `userdata.get()` โดยไม่แสดงค่า key

รันเซลล์จากบนลงล่าง หลังติดตั้ง package และเตรียม key แล้ว ทดสอบตัวอย่างหลักใน Colab ด้วย API แล้ว แต่ผล LLM อาจเปลี่ยนได้ การผ่านตัวอย่างนี้ไม่ใช่การรับรองทุกคำถามหรือการใช้งานจริง

ใช้ `gpt-4.1-mini` สำหรับสร้างคำตอบ (ไม่เกิน 500 tokens ต่อ request) และ `text-embedding-3-small` สำหรับ Embedding โดย Agent Loop จำกัดไว้ 5 รอบ

In [1]:
%pip -q install openai numpy

In [2]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL = "gpt-4.1-mini"

In [3]:
chunks = [
    "[SOP-1 รับน้ำยาง] บันทึก Lot ผู้ขาย น้ำหนักสุทธิ ทะเบียนรถ และเวลารับเข้า เก็บรหัสผู้ขายเพื่อเชื่อมโยงข้อมูล Traceability",
    "[SOP-2 QC] เก็บผล DRC พร้อมรหัสตัวอย่างและ Lot หากผลยังไม่ครบ ให้ระบุสถานะรอ QC ห้ามสรุปว่าผ่านเกณฑ์จากข้อมูลที่หายไป",
    "[SOP-3 คลังสินค้า] Lot ที่รอ QC ต้องแยกในพื้นที่พักรอผลและติดป้าย Lot ตรวจสถานะก่อนยืนยันการจัดส่ง",
    "[SOP-4 ประสานงาน] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 30 ซม. ให้ประสานหัวหน้าคลังและฝ่ายขนส่งเพื่อตรวจสอบแผนรับส่งสินค้า",
    "[SOP-5 น้ำท่วม] ในสถานการณ์ฝึกอบรม ระดับน้ำตั้งแต่ 50 ซม. ให้พักแผนโหลดสินค้าของพื้นที่ที่ได้รับผลกระทบ แจ้งหัวหน้าคลัง ฝ่ายขนส่ง และ EHS เพื่อประเมินตามแผนของพื้นที่",
    "[SOP-6 ส่งมอบ] เมื่อเวลารับส่งเปลี่ยน ให้ฝ่ายขนส่งประสานฝ่ายขาย แจ้งกำหนดการใหม่ที่ยืนยันแล้ว ห้ามรับรองเวลาส่งมอบที่ยังไม่ยืนยัน",
    "[SOP-7 ซ่อมบำรุง] เมื่อพบปัญหาเครื่องจักร ให้บันทึกอาการ รหัสเครื่อง และเวลา แล้วส่งใบงานให้ผู้รับผิดชอบซ่อมบำรุง",
    "[SOP-8 สิ่งแวดล้อม] หากพบข้อร้องเรียนด้านกลิ่นหรือน้ำเสีย ให้บันทึกเวลา จุดที่พบ และรายละเอียด ส่งต่อผู้รับผิดชอบสิ่งแวดล้อมเพื่อตรวจสอบ"
]

In [4]:
import numpy as np

EMBED_MODEL = "text-embedding-3-small"

def embed(texts):
    result = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([item.embedding for item in result.data])

vectors = embed(chunks)
print("ขนาด Embedding matrix:", vectors.shape)

ขนาด Embedding matrix: (8, 1536)


In [5]:
def retrieve(question, k=2):
    query = embed([question])[0]
    scores = vectors @ query / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query))
    indices = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in indices]

In [6]:
question = "Lot ที่ยังไม่มีผล QC ควรจัดเก็บและตรวจสอบอย่างไร?"
for text, score in retrieve(question, k=3):
    print(round(score, 3), text)

0.555 [SOP-3 คลังสินค้า] Lot ที่รอ QC ต้องแยกในพื้นที่พักรอผลและติดป้าย Lot ตรวจสถานะก่อนยืนยันการจัดส่ง
0.531 [SOP-2 QC] เก็บผล DRC พร้อมรหัสตัวอย่างและ Lot หากผลยังไม่ครบ ให้ระบุสถานะรอ QC ห้ามสรุปว่าผ่านเกณฑ์จากข้อมูลที่หายไป
0.39 [SOP-8 สิ่งแวดล้อม] หากพบข้อร้องเรียนด้านกลิ่นหรือน้ำเสีย ให้บันทึกเวลา จุดที่พบ และรายละเอียด ส่งต่อผู้รับผิดชอบสิ่งแวดล้อมเพื่อตรวจสอบ


## ลองทำด้วยตนเอง
1. ถามเรื่องการพัก Lot ที่ยังไม่มีผล QC ด้วยคำอื่น
2. เปลี่ยน k จาก 1 เป็น 3 แล้วเปรียบเทียบข้อความที่พบ
3. ถามเรื่องที่ไม่มีใน SOP เช่น สวัสดิการพนักงาน อันดับสูงสุดแปลว่ามีคำตอบจริงหรือไม่?